In [ ]:
#import pandas as pd
#import numpy as np
##surpress divide warnings
#np.errstate(invalid='ignore', divide='ignore')
#import matplotlib.pyplot as plt
#import pickle
#import pickle as pkl
#from src.data_tools.get_data import get_data
#from scipy.stats import norm
#
#from src.plotting_tools.cms_format import cms_format_fig, cms_style
#cms_style()

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import pickle as pkl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
from src.plotting_tools.cms_format import cms_format_fig, cms_style
cms_style()
from src.assets.output_dir import output_dir
output_dir
from src.plotting_tools.Bins import Bins, bins
from src.plotting_tools.SysHist import SysHist
from src.data_tools.StackPlotter import get_stack_plotter

In [ ]:
era = "2017"
ismc = 0
xrange = (120,401)

In [ ]:
outname="{}/abcd/abcd_dict_data_{}_ismc{}_v2.pkl".format(output_dir, era, ismc)
with open(outname,'rb') as f:
    abcd = pkl.load(f)

In [ ]:
##
## create fit variations
##

In [ ]:
CR13_hist = SysHist.from_dict(abcd['CR13'])
hist = CR13_hist.reduce_range(*xrange)
hist_rebin = hist.rebin(bins.bin_edges).reduce_range(*xrange)
fig, ax = plt.subplots()
hist_density = hist.make_density_hist()
hist_density.draw(ax)

In [ ]:
from src.general.functions import make_bpoly, lognorm, log_norm_np, log_norm_unp
from scipy.optimize import curve_fit
import copy 
from src.plotting_tools.utils import rebin_np


In [ ]:
def make_variations(_nBin, array):
    x = array[_nBin]
    std_dev = x**.5
    up, down = copy.copy(array), copy.copy(array)
    up[_nBin]+=std_dev
    down[_nBin]-=std_dev
    return up, down
    

In [ ]:
def fit(_x, _y):
    total_events = _y.sum()
    popt, pcov = curve_fit(ln_function, _x,_y,
                       p0=[total_events*3, .8, 80, 70], bounds = ([0, .2, 50, 50], [total_events*100, 1, 100, 100])
                          )
    return popt, pcov

def lognorm_variable_width(*args, widths=1):
    y = log_norm_np(*args)
    return y*widths

ln_function = lambda *x: lognorm_variable_width(*x, widths=hist.calc_bin_widths())    
   

In [ ]:
x = hist_density.calc_bin_centers()
xp = hist_rebin.calc_bin_centers()
bin_widths = hist_rebin.calc_bin_widths()
ratios_CR13 = {}
for i in range(len(hist_density.nominal)):
    up, down = make_variations(i, hist.nominal)
    #make fits
    poptup, pcov = fit(x, up)                                                               
    poptdown, pcov = fit(x, down)                                                               
    poptnom, pcov = fit(x, hist.nominal) 
    
    up_fit = ln_function(x, *poptup)
    down_fit = ln_function(x, *poptdown)
    nom_fit = ln_function(x, *poptnom)
    
    up_fit = rebin_np(x, hist_rebin.bin_edges, up_fit) 
    down_fit = rebin_np(x, hist_rebin.bin_edges, down_fit) 
    nom_fit = rebin_np(x, hist_rebin.bin_edges, nom_fit) 

    
    #make_plot
    if (i % 10)==0:
        print(i)
        fig, axs = plt.subplots(2,1)
        top, bottom = axs
        
        top.plot(x, up, label='Up', color='blue', ls=":")
        top.plot(xp, up_fit/bin_widths, label='Up Fit', color='blue')
        
        top.plot(x, down, label='Down', color='red', ls=":")
        top.plot(xp, down_fit/bin_widths, label='Down Fit', color='red')
        
        top.plot(x, hist.nominal, color='black', ls=":")
        top.plot(xp, nom_fit/bin_widths, label='Nom Fit', color='black')
        
        bottom.plot(xp, up_fit/nom_fit, color='blue')
        bottom.plot(xp, nom_fit/nom_fit, color='black')
        bottom.plot(xp, down_fit/nom_fit, color='red')
        bottom.set_ylim(.9,1.1)
        
        top.legend(title='Bin: {}'.format(i))
        top.set_xlabel('$m_{\ell\ell}$ [GeV]')
        top.set_ylabel('Counts')
        cms_format_fig(era, top, "\emph{Preliminary}")

    ratios_CR13[i] = {'up': up_fit/nom_fit, "down": down_fit/nom_fit}

In [ ]:
CR23_hist = SysHist.from_dict(abcd['CR23'])
hist = CR23_hist.reduce_range(*xrange)
hist_rebin = hist.rebin(bins.bin_edges).reduce_range(119,401)
fig, ax = plt.subplots()
hist_density = hist.make_density_hist()
hist_density.draw(ax)

In [ ]:
x = hist_density.calc_bin_centers()
xp = hist_rebin.calc_bin_centers()
bin_widths = hist_rebin.calc_bin_widths()
ratios_CR23 = {}
for i in range(len(hist_density.nominal)):
    up, down = make_variations(i, hist.nominal)
    #make fits
    poptup, pcov = fit(x, up)                                                               
    poptdown, pcov = fit(x, down)                                                               
    poptnom, pcov = fit(x, hist.nominal) 
    
    up_fit = ln_function(x, *poptup)
    down_fit = ln_function(x, *poptdown)
    nom_fit = ln_function(x, *poptnom)
    
    up_fit = rebin_np(x, hist_rebin.bin_edges, up_fit) 
    down_fit = rebin_np(x, hist_rebin.bin_edges, down_fit) 
    nom_fit = rebin_np(x, hist_rebin.bin_edges, nom_fit) 

    
    #make_plot
    if (i % 10)==0:
        print(i)
        fig, axs = plt.subplots(2,1)
        top, bottom = axs
        
        top.plot(x, up, label='Up', color='blue', ls=":")
        top.plot(xp, up_fit/bin_widths, label='Up Fit', color='blue')
        
        top.plot(x, down, label='Down', color='red', ls=":")
        top.plot(xp, down_fit/bin_widths, label='Down Fit', color='red')
        
        top.plot(x, hist.nominal, color='black', ls=":")
        top.plot(xp, nom_fit/bin_widths, label='Nom Fit', color='black')
        
        bottom.plot(xp, up_fit/nom_fit, color='blue')
        bottom.plot(xp, nom_fit/nom_fit, color='black')
        bottom.plot(xp, down_fit/nom_fit, color='red')
        bottom.set_ylim(.9,1.1)
        
        top.legend(title='Bin: {}'.format(i))
        top.set_xlabel('$m_{\ell\ell}$ [GeV]')
        top.set_ylabel('Counts')
        cms_format_fig(era, top, "\emph{Preliminary}")

    ratios_CR23[i] = {'up': up_fit/nom_fit, "down": down_fit/nom_fit}

In [ ]:
##
## create data hists
##
sp = get_stack_plotter(output_dir, era, bins='none')

In [ ]:
fig, ax = plt.subplots()
sp.binning = bins.bin_edges
sp.x_range = xrange

hist_SR1 = sp.draw_background(ax, 'DiLepMass', 'SR1')
hist_SR1_inv = hist_SR1.inverse_make_density_hist()
hist_SR1_inv.draw(ax)

data_SR1 = hist_SR1_inv.nominal

In [ ]:
fig, ax = plt.subplots()
sp.binning = bins.bin_edges
sp.x_range = (120, 401)

hist_SR2 = sp.draw_background(ax, 'DiLepMass', 'SR2')
hist_SR2_inv = hist_SR2.inverse_make_density_hist()
hist_SR2_inv.draw(ax)

data_SR2 = hist_SR2_inv.nominal

In [ ]:
##
## create workspace
##

In [ ]:
csvname = f'{output_dir}/combine_data/{era}/{era}_abcd_shapes_df_input.csv'
csvname

In [ ]:
def make_list(_list, channel, process, systematic, values, masses):
    for i, (x, m) in enumerate(zip(values, masses)):
        _list.append({'channel': channel, 'process': process,
                      'systematic': systematic, 'bin':m, 'sum_w': x, 'sum_ww':x})

In [ ]:
abcd['SR1']['bins']

In [ ]:
masses = Bins(abcd['SR1']['bins']).calc_bin_centers()

In [ ]:
csv_list = []

hist = abcd['SR1']['nom']
make_list(csv_list, 'SR1', 'ABCD', 'nominal', hist, masses)

hist = abcd['SR2']['nom']
make_list(csv_list, 'SR2', 'ABCD', 'nominal', hist, masses)

In [ ]:

for key, values in ratios_CR23.items():
    up, down = values['up'], values['down']
    make_list(csv_list, 'SR2', 'ABCD', f'fit_{key}_Up', up, masses)
    make_list(csv_list, 'SR2', 'ABCD', f'fit_{key}_Down', down, masses)
    
for key, values in ratios_CR13.items():
    up, down = values['up'], values['down']
    make_list(csv_list, 'SR1', 'ABCD', f'fit_{key}_Up', up, masses)
    make_list(csv_list, 'SR1', 'ABCD', f'fit_{key}_Down', down, masses)

In [ ]:
#data hists
make_list(csv_list, 'SR1', 'data_obs', 'nominal', data_SR1, masses)
make_list(csv_list, 'SR2', 'data_obs', 'nominal', data_SR2, masses)


In [ ]:
df = pd.DataFrame(csv_list)

df.to_csv(csvname, index=False)